In [2]:
import pandas as pd

df = pd.read_csv("../IMDB Dataset.csv")
print(df.head())
print(df.columns)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
Index(['review', 'sentiment'], dtype='str')


In [3]:
df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

In [4]:
df.head()

,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


# Create PyTorch Dataset

In [10]:
import torch
from torch.utils.data import Dataset


class MyDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_length=256):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length


    def __len__(self):
        return len(self.data)


    def __getitem__(self, idx):

        review = self.data.iloc[idx]["review"]
        label = self.data.iloc[idx]["label"]

        tokens = self.tokenizer(
            review,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }

# Create tokenizer

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

# Create dataset and dataloader

In [ ]:
from torch.utils.data import DataLoader

dataset = MyDataset(
    df,
    tokenizer
)


loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True
)

In [21]:
dataset.__len__()

50000

In [18]:
dataset.data

,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1
...,...,...,...
49995,I thought this movie did a down right good job...,positive,1
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative,0
49997,I am a Catholic taught in parochial elementary...,negative,0
49998,I'm going to have to disagree with the previou...,negative,0


# Test one batch

In [13]:
batch = next(iter(loader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["label"])

torch.Size([16, 256])
torch.Size([16, 256])
tensor([1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1])


In [9]:
batch

{'input_ids': tensor([[  101,  2984, 15030,  ...,  2021,  2009,   102],
         [  101,  4962, 20578,  ...,  4099, 10214,   102],
         [  101,  2028,  1997,  ...,     0,     0,     0],
         ...,
         [  101,  2023,  2003,  ...,     0,     0,     0],
         [  101,  2023,  3185,  ...,  2320,  1996,   102],
         [  101,  2035,  1999,  ...,  1998, 23069,   102]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]]),
 'label': tensor([1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0])}

In [19]:
for batch in loader:
    print(batch)

{'input_ids': tensor([[ 101, 2339, 1996,  ..., 2000, 2037,  102],
        [ 101, 2023, 3185,  ..., 4332, 2601,  102],
        [ 101, 2990, 8379,  ...,    0,    0,    0],
        ...,
        [ 101, 3302, 2001,  ...,    0,    0,    0],
        [ 101, 1000, 6802,  ...,    0,    0,    0],
        [ 101, 2026, 6707,  ..., 2012, 2560,  102]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1]]), 'label': tensor([1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0])}
{'input_ids': tensor([[  101,  2002,  3248,  ...,     0,     0,     0],
        [  101,  1045,  2288,  ...,     0,     0,     0],
        [  101,  1996,  2143,  ...,     0,     0,     0],
        ...,
        [  101,  1998,  1996,  ...,  2156, 14744,   102],
        [  101,  2023,  2003,  ...,  1037,  7656,   102],
        [  101,  1996, 16913,  ...,   